# 03. 128개 Expert 정렬 모델 만들기

원본 128개 Expert를 모두 유지하면서 층별로 순서를 바꿉니다. LCB 첫 100문항의 decode 단계에서 선택된 횟수를 합산해 내림차순으로 정렬하며, 횟수가 같으면 원래 ID가 작은 순서입니다. 선택 횟수는 기여도나 정확도가 아닙니다. 생성되는 manifest에 층별 원래 ID 대응 관계가 저장됩니다.

다음 04번 노트북은 이 정렬 모델에서 0~103번을 남깁니다. 파일 바이트 검증은 추론 동등성이나 품질을 보장하지 않습니다.

In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd().resolve()
if ROOT.name == 'ipynb': ROOT = ROOT.parent
assert (ROOT / 'moe_prune').is_dir(), '저장소 루트 또는 ipynb 폴더에서 실행하세요.'
sys.path.insert(0, str(ROOT))
from tools import reorder_experts as reorder

RUN_REORDER = False
ORIGINAL_MODEL = reorder.SOURCE
ORDERED_MODEL = reorder.OUTPUT
ORDERED_MANIFEST = reorder.MANIFEST

print({'원본 모델': str(ORIGINAL_MODEL), '있음': ORIGINAL_MODEL.is_file()})
print({'정렬 모델 출력': str(ORDERED_MODEL), '이미 있음': ORDERED_MODEL.exists()})
print({'정렬 manifest 출력': str(ORDERED_MANIFEST), '이미 있음': ORDERED_MANIFEST.exists()})
print({'사용 기록 폴더': str(reorder.PROFILE), '있음': reorder.PROFILE.is_dir()})
print({'데이터 폴더': str(reorder.INPUT), '있음': reorder.INPUT.is_dir()})

## 정렬 모델 생성

기존 `download/lcb-first100-results/expert-usage`의 `index.json`, `001.json`~`100.json`과 `download/lcb-first100-results` 아래 문제별 `result.json`이 필요합니다. `download/gemma4-lcb-first100-local-grading`의 `records.jsonl`, `generations.jsonl`, `bundle.json`, `generation-identity.json`도 필요합니다. 실행 시 기록의 출처와 원본 모델 해시를 검증합니다. 다운로드나 새 추론은 실행하지 않습니다.

`RUN_REORDER=True`로 바꾼 뒤 셀을 순서대로 실행하세요. 원본과 정렬 모델 각각에 필요한 디스크 공간 및 전체 해시·바이트 검증 시간이 필요합니다. 출력 모델이나 manifest가 이미 있으면 중단하므로 다시 만들 때는 `ORDERED_MODEL`과 `ORDERED_MANIFEST`를 모두 사용하지 않은 경로로 바꾸세요. 04번의 104개 모델 생성은 정해진 정렬 모델 해시를 검증합니다.

In [ ]:
if RUN_REORDER:
    result = reorder.export(source=ORIGINAL_MODEL, output=ORDERED_MODEL, manifest=ORDERED_MANIFEST)
    print({key: result[key] for key in ('output_path', 'output_bytes', 'output_sha256', 'verification', 'limitation')})
else:
    print('정렬 모델 생성 대기 중: RUN_REORDER=True로 설정하세요.')